# Real-document evaluation

Generated passports show what DocuLens can do in principle. This notebook shows how it does on
**16 real-world sample images**: government specimen passports, stock sample passports, an ID card,
and phone photos, including very small and blurred ones.

Each image was scanned through the running app and **every extracted field was checked by hand**
against the image. A missing field counts as wrong. Fields not printed on the document, or
deliberately blurred out, are not counted. The scores are in
[`real-document-review.csv`](real-document-review.csv).

> To protect privacy, samples are identified only by number and country. Names, document numbers
> and the images themselves are not published.

In [1]:
from report_helpers import *  # noqa: F403  (paths and small display helpers)

In [2]:
import csv

with open(HERE / "real-document-review.csv", encoding="utf-8") as handle:
    review = list(csv.DictReader(handle))
correct = sum(int(r["correct"]) for r in review)
scored = sum(int(r["scored"]) for r in review)
perfect = sum(r["correct"] == r["scored"] for r in review)
print(f"{correct}/{scored} fields correct ({correct / scored:.0%}); {perfect} of {len(review)} documents fully correct")
review.sort(key=lambda r: int(r["correct"]) / int(r["scored"]), reverse=True)
table(["Sample", "Document", "Image quality", "Correct", "Score", "Notes"],
      [[r["sample"], r["document"], r["image_quality"], f"{r['correct']}/{r['scored']}",
        bar(int(r["correct"]) / int(r["scored"]), 20), r["notes"]] for r in review])

172/215 fields correct (80%); 5 of 16 documents fully correct


| Sample | Document | Image quality | Correct | Score | Notes |
|---|---|---|---|---|---|
| 01 | USA passport | Good photo | 14/14 | `████████████████████ 100.0%` | All fields correct |
| 02 | USA passport | Good photo | 14/14 | `████████████████████ 100.0%` | All fields correct |
| 05 | USA passport | Good photo, tilted | 14/14 | `████████████████████ 100.0%` | All fields correct |
| 07 | UAE resident ID card | Screenshot, front and back | 12/12 | `████████████████████ 100.0%` | All fields correct |
| 15 | Switzerland passport | Official specimen | 13/13 | `████████████████████ 100.0%` | All fields correct |
| 06 | USA passport | Sample image | 12/14 | `█████████████████░░░ 85.7%` | Authority shortened; issuing country code missing |
| 14 | Bangladesh passport | Phone photo, two pages | 12/14 | `█████████████████░░░ 85.7%` | Full name taken from father's name; authority missing |
| 16 | World Passport | Scan, number blurred out | 11/13 | `█████████████████░░░ 84.6%` | Caption text returned as passport number; full name taken from instruction text |
| 09 | Greece passport | Very small (329 x 232 px) | 11/14 | `████████████████░░░░ 78.6%` | Place of birth issue date and authority missing |
| 10 | Greece passport | Same image as 09 | 11/14 | `████████████████░░░░ 78.6%` | Place of birth issue date and authority missing |
| 12 | USA passport | Sample with damaged MRZ | 11/14 | `████████████████░░░░ 78.6%` | Passport number and expiry wrong; authority missing |
| 11 | Slovakia passport | Official specimen | 10/14 | `██████████████░░░░░░ 71.4%` | Issue and expiry dates wrong; place of birth and authority missing |
| 03 | Ireland passport | Heavily blurred | 9/14 | `█████████████░░░░░░░ 64.3%` | Apostrophe dropped from surname and full name; place of birth and authority missing; issue date wrong |
| 13 | US refugee travel document | Very small (330 x 222 px) | 6/10 | `████████████░░░░░░░░ 60.0%` | Labelled as passport; sex missing; expiry not converted to a date |
| 04 | India passport | Tiny (214 x 148 px) | 7/14 | `██████████░░░░░░░░░░ 50.0%` | Surname misread; nationality sex place of birth place of issue and issue date missing |
| 08 | India passport | Blurred and pixelated | 5/13 | `████████░░░░░░░░░░░░ 38.5%` | Surname misread; nationality dates and places missing |

## What goes wrong, and why

| Problem | Seen in | Cause |
|---|---|---|
| Issuing authority missing | 03, 09, 10, 11, 12, 14 | Small multilingual captions are misread, so the value has no heading to attach to |
| Place of birth missing | 03, 04, 08, 09, 10, 11 | Same cause, plus very small text |
| Wrong or missing dates | 03, 08, 11, 12, 13 | Blur, tiny text, or a damaged MRZ that cannot confirm the printed date |
| Surname misread | 04, 08 | Tiny or pixelated image |
| Full name built from the wrong text | 14, 16 | A nearby "Name" caption for another person (father) or instruction text |
| Caption text returned as the passport number | 16 | The real number was blurred out, so the nearest text was taken |
| Refugee travel document labelled as a passport | 13 | The document type code "TP" is treated as a passport |

## What works well

- Clear photos of standard passports are read completely, including tilted ones.
- The MRZ, when readable, confirms names, numbers and dates, and catches OCR mistakes.
- Captions in several languages (English, French, Dutch, Greek, Slovak, Arabic, Bengali) are understood.
- Uncertain or conflicting fields are clearly marked for a person to check.

## Scan your own documents

Set `DOCULENS_SAMPLE_DIR` to a folder of passport or ID images (or PDFs) before starting Jupyter.
Each file is scanned locally and its extracted fields are listed so you can check them by hand.
**Clear this cell's output before sharing the notebook if you use real documents.**

In [3]:
sample_dir = os.environ.get("DOCULENS_SAMPLE_DIR")
if not sample_dir:
    print("DOCULENS_SAMPLE_DIR is not set; skipping.")
else:
    sys.path.insert(0, BACKEND)
    os.chdir(BACKEND)
    from app.core.config import Settings
    from app.core.store import ScanStore
    from app.schemas.models import ScanOptions
    from app.services.ocr import OCRService
    from app.services.pipeline import run_scan

    engine = OCRService()
    engine.warmup()
    files = sorted(p for p in Path(sample_dir).iterdir()
                   if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".pdf"})
    for number, path in enumerate(files, 1):
        store = ScanStore(60, 1, 160 * 1024 * 1024)
        job = store.create("notebook")
        run_scan(job, {"front": path.read_bytes()}, ScanOptions(date_order="dmy"), store, engine, Settings())
        result = job.result
        print(f"\n{number:02d}. {path.name}: {job.status}, document type {result.document_type if result else '-'}")
        for field in ([*result.fields.values(), *result.other_fields] if result else []):
            if field.value:
                print(f"    {field.field_name:20s} {field.display_value or field.value}  [{field.status}]")
        store.clear()
    os.chdir(HERE)

DOCULENS_SAMPLE_DIR is not set; skipping.
